# 01 — Exploração dos Vídeos do Cliente (EDA)

> **Ponto de partida do projeto.** Execute este notebook primeiro para entender o que o cliente enviou.

**Perguntas que este notebook responde para o time:**

| Pergunta | Onde ver |
|---|---|
| Quantos vídeos e de quais locais? | Seção 4 |
| Qual o tamanho total e espaço em disco? | Seção 4 |
| Qual a duração dos vídeos (curtos, longos)? | Seção 5 |
| Quais resoluções e formatos (mp4, avi)? | Seção 5 |
| Em quais horários há mais gravação? | Seção 6 |
| Quais câmeras gravam mais? | Seção 6 |
| Tem muita cena parada (sem movimento)? | Seção 7 |
| Quais vídeos têm mais atividade? | Seção 7 |
| Como são os frames do vídeo visualmente? | Seção 8 |

**Suporte a formatos:** `.mp4`, `.avi`, `.mov`, `.mkv`, `.ts`, `.mts`, `.m4v`
— incluindo **AVI como sequência de imagens**.

**Padrão de nome esperado:** `LocalCidade_chNumero_YYYYMMDDHHmmss_YYYYMMDDHHmmss.ext`  
Ex: `CentroSP_ch03_20240315143022_20240315153022.mp4`  
O parser tem fallback para nomes fora do padrão.

---
**Saídas geradas em `data/client_analysis/`:**
- `catalog.xlsx` — catálogo completo
- `summary_by_local.csv` — resumo por local
- `thumbnails/<video>.jpg` — grid visual de cada vídeo
- `heatmap_local_hora.png` — mapa de calor local × hora
- `motion_profiles.json` — perfil de movimento (usado no notebook 02)

**A seguir:** ver `README.md` na raiz para a ordem recomendada do pipeline (inclui dataset público e pré-treino antes do `02`).

## 1. Instalação

In [ ]:
!pip install -q opencv-python-headless matplotlib seaborn pandas openpyxl tqdm ipywidgets Pillow

## 2. Imports

In [ ]:
import os, re, cv2, json, math, warnings, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
from tqdm.notebook import tqdm
from IPython.display import display, HTML
from PIL import Image as PILImage

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
sns.set_style("whitegrid")


@contextlib.contextmanager
def _suppress_stderr():
    """Silencia stderr no nível do OS — elimina logs C/C++ do decoder HEVC/H264 do OpenCV."""
    devnull = os.open(os.devnull, os.O_WRONLY)
    old_stderr = os.dup(2)
    os.dup2(devnull, 2)
    os.close(devnull)
    try:
        yield
    finally:
        os.dup2(old_stderr, 2)
        os.close(old_stderr)

## 3. Configuração — aponte para a pasta do cliente

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CONFIGURE AQUI — apenas este bloco precisa ser alterado
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

CLIENT_VIDEO_DIR = Path("../data/client_videos")   # pasta com os vídeos do cliente
OUTPUT_DIR       = Path("../data/client_analysis") # onde salvar análises

# Padrão do nome: Local_Camera_TimestampInicio_TimestampFim
# Ex: CentroSP_ch03_20240315143022_20240315153022.mp4
FILENAME_PATTERN = re.compile(
    r"^(?P<local>[A-Za-z0-9]+)_"
    r"(?P<camera>ch\d+)_"
    r"(?P<ts_start>\d{14})_"
    r"(?P<ts_end>\d{14})",
    re.IGNORECASE,
)
TS_FORMAT = "%Y%m%d%H%M%S"

# Análise de movimento
MOTION_SAMPLE_FPS   = 1      # 1 frame/s para análise rápida
MOTION_THRESHOLD    = 15.0   # diff médio para considerar movimento

# Thumbnails
MAX_THUMBS_PER_VIDEO = 8
THUMB_SIZE           = (320, 180)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "thumbnails").mkdir(exist_ok=True)

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".ts", ".mts", ".m4v"}

print(f"Pasta do cliente : {CLIENT_VIDEO_DIR.resolve()}")
print(f"Saída            : {OUTPUT_DIR.resolve()}")

## 4. Varredura e leitura de metadados

In [ ]:
# ── Funções de leitura ────────────────────────────────────────────────────────

def is_avi_image_sequence(path: Path) -> bool:
    """Detecta se um AVI é tratado como sequência de imagens (sem stream de vídeo)."""
    if path.suffix.lower() != ".avi":
        return False
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return True  # não abre → provavelmente imagem
    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()
    return fps <= 1 and total <= 1


def open_as_image(path: Path) -> np.ndarray | None:
    """Tenta abrir arquivo como imagem estática (PNG, AVI-imagem, etc.)."""
    try:
        img = cv2.imread(str(path))
        if img is not None:
            return img
        pil = PILImage.open(path)
        return cv2.cvtColor(np.array(pil.convert("RGB")), cv2.COLOR_RGB2BGR)
    except Exception:
        return None


def parse_filename(path: Path) -> dict:
    """Extrai local, câmera e timestamps do nome do arquivo."""
    stem = path.stem
    m    = FILENAME_PATTERN.match(stem)
    local = camera = ts_start = ts_end = dt_start = dt_end = dur_name = None

    if m:
        local, camera = m.group("local"), m.group("camera")
        try:
            dt_start = datetime.strptime(m.group("ts_start"), TS_FORMAT)
            dt_end   = datetime.strptime(m.group("ts_end"),   TS_FORMAT)
            ts_start = dt_start.isoformat(sep=" ")
            ts_end   = dt_end.isoformat(sep=" ")
            dur_name = (dt_end - dt_start).total_seconds()
        except ValueError:
            pass
    else:
        # Fallback: extrai o que consegue
        parts = stem.split("_")
        local  = parts[0] if parts else stem[:20]
        camera = parts[1] if len(parts) > 1 else None
        for ts in re.findall(r"\d{14}", stem):
            try:
                dt = datetime.strptime(ts, TS_FORMAT)
                if dt_start is None: dt_start, ts_start = dt, dt.isoformat(sep=" ")
                elif dt_end is None:
                    dt_end, ts_end = dt, dt.isoformat(sep=" ")
                    dur_name = (dt_end - dt_start).total_seconds()
            except ValueError:
                pass

    return dict(local=local, camera=camera, ts_start=ts_start, ts_end=ts_end,
                dt_start=dt_start, dt_end=dt_end, duration_from_name=dur_name,
                pattern_matched=m is not None)


def get_video_props(path: Path) -> dict:
    """Propriedades técnicas via OpenCV."""
    size_mb = round(path.stat().st_size / 1e6, 2)
    base    = dict(file_size_mb=size_mb, width=None, height=None, fps=None,
                   frame_count=None, duration_sec=None, duration_str=None,
                   codec=None, readable=False, is_image=False)

    # Tenta como imagem primeiro (AVI-imagem ou frame único)
    if is_avi_image_sequence(path):
        img = open_as_image(path)
        if img is not None:
            h, w = img.shape[:2]
            base.update(width=w, height=h, fps=0, frame_count=1,
                        duration_sec=0, duration_str="imagem",
                        codec="IMAGE", readable=True, is_image=True)
        return base

    try:
        with _suppress_stderr():
            cap = cv2.VideoCapture(str(path))
        if not cap.isOpened():
            return base
        w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        nf  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fcc = int(cap.get(cv2.CAP_PROP_FOURCC))
        codec = "".join(chr((fcc >> (8*i)) & 0xFF) for i in range(4)).strip() or "?"
        cap.release()
        dur = nf / fps if fps and fps > 0 and nf > 0 else None
        base.update(width=w, height=h, fps=round(fps,2), frame_count=nf,
                    duration_sec=round(dur,1) if dur else None,
                    duration_str=str(timedelta(seconds=int(dur))) if dur else None,
                    codec=codec, readable=True)
    except Exception as e:
        base["error"] = str(e)
    return base


# ── Varredura ─────────────────────────────────────────────────────────────────
all_paths = sorted([p for p in CLIENT_VIDEO_DIR.rglob("*")
                    if p.suffix.lower() in VIDEO_EXTENSIONS])

if not all_paths:
    raise FileNotFoundError(
        f"Nenhum vídeo encontrado em {CLIENT_VIDEO_DIR.resolve()}\n"
        "Coloque os vídeos do cliente nessa pasta e reexecute."
    )

print(f"Arquivos encontrados: {len(all_paths)}")
ext_counts = pd.Series([p.suffix.lower() for p in all_paths]).value_counts()
for ext, n in ext_counts.items():
    print(f"  {ext:8s}: {n}")

In [ ]:
# ── Montar catálogo ───────────────────────────────────────────────────────────
rows = []
for p in tqdm(all_paths, desc="Lendo metadados"):
    row = {"filename": p.name, "filepath": str(p), "extension": p.suffix.lower()}
    row.update(parse_filename(p))
    row.update(get_video_props(p))
    row["tag"] = f"{row['local']}_{row['camera']}" if row.get("local") and row.get("camera") else p.stem[:25]
    rows.append(row)

df = pd.DataFrame(rows)

n_videos  = (df["is_image"] == False).sum()
n_images  = df["is_image"].sum()
n_unread  = (~df["readable"]).sum()
total_gb  = df["file_size_mb"].sum() / 1024
total_h   = df[df["is_image"]==False]["duration_sec"].dropna().sum() / 3600

display(HTML(f"""
<div style='background:#eef6ff;padding:14px;border-radius:8px;font-size:14px;line-height:2'>
<b>Resumo rápido</b><br>
📹 Vídeos       : <b>{n_videos}</b><br>
🖼️  Imagens (AVI) : <b>{n_images}</b><br>
⚠️  Ilegíveis    : <b>{n_unread}</b><br>
💾 Tamanho total : <b>{total_gb:.2f} GB</b><br>
⏱️  Horas de gravação : <b>{total_h:.1f} h</b><br>
📍 Locais únicos : <b>{df['local'].nunique()}</b><br>
📷 Câmeras únicas: <b>{df['camera'].nunique()}</b>
</div>
"""))

# Salvar
save_cols = ["filename","extension","local","camera","tag","ts_start","ts_end",
             "duration_from_name","file_size_mb","width","height","fps",
             "frame_count","duration_sec","duration_str","codec",
             "readable","is_image","pattern_matched","filepath"]
df[[c for c in save_cols if c in df.columns]].to_csv(OUTPUT_DIR/"catalog.csv", index=False)
df[[c for c in save_cols if c in df.columns]].to_excel(OUTPUT_DIR/"catalog.xlsx", index=False)
print(f"\n✓ Catálogo salvo: {OUTPUT_DIR/'catalog.xlsx'}")

## 5. Distribuição de formatos, duração e tamanho

In [ ]:
df_vid = df[df["is_image"] == False].copy()
df_img = df[df["is_image"] == True].copy()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1 — Extensões
ext_vc = df["extension"].value_counts()
axes[0,0].pie(ext_vc.values, labels=ext_vc.index, autopct="%1.0f%%",
              colors=sns.color_palette("Set2", len(ext_vc)))
axes[0,0].set_title("Formatos de arquivo", fontsize=11)

# 2 — Duração dos vídeos
if df_vid["duration_sec"].notna().any():
    dur_min = df_vid["duration_sec"].dropna() / 60
    axes[0,1].hist(dur_min, bins=20, color="steelblue", edgecolor="black", alpha=0.85)
    axes[0,1].axvline(dur_min.mean(), color="red", lw=2, ls="--",
                      label=f"Média: {dur_min.mean():.1f} min")
    axes[0,1].set_title("Duração dos vídeos", fontsize=11)
    axes[0,1].set_xlabel("Minutos")
    axes[0,1].legend(fontsize=9)

# 3 — Tamanho dos arquivos
axes[0,2].hist(df["file_size_mb"], bins=25, color="mediumseagreen",
               edgecolor="black", alpha=0.85)
axes[0,2].set_title("Tamanho dos arquivos", fontsize=11)
axes[0,2].set_xlabel("MB")

# 4 — Resoluções
if df_vid["width"].notna().any():
    df_vid["resolucao"] = (df_vid["width"].astype(str) + "×" + df_vid["height"].astype(str))
    res_vc = df_vid["resolucao"].value_counts().head(8)
    res_vc.plot.barh(ax=axes[1,0], color="darkorange", edgecolor="black", alpha=0.85)
    axes[1,0].set_title("Resoluções (vídeos)", fontsize=11)

# 5 — FPS
if df_vid["fps"].notna().any():
    fps_vc = df_vid["fps"].value_counts().head(8)
    fps_vc.plot.bar(ax=axes[1,1], color="slateblue", edgecolor="black", alpha=0.85)
    axes[1,1].set_title("FPS dos vídeos", fontsize=11)
    axes[1,1].tick_params(axis="x", rotation=0)

# 6 — Horas por local
if "local" in df_vid.columns and df_vid["local"].notna().any():
    horas = df_vid.groupby("local")["duration_sec"].sum().dropna() / 3600
    horas.sort_values().plot.barh(ax=axes[1,2], color="cadetblue",
                                  edgecolor="black", alpha=0.85)
    axes[1,2].set_title("Horas gravadas por local", fontsize=11)
    axes[1,2].set_xlabel("Horas")

plt.suptitle("Visão Geral dos Vídeos do Cliente", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"eda_overview.png", bbox_inches="tight")
plt.show()

# Tabela de imagens AVI
if len(df_img) > 0:
    print(f"\n⚠ {len(df_img)} arquivo(s) identificados como IMAGEM (não vídeo):")
    display(df_img[["filename","file_size_mb","width","height"]].head(10))

## 6. Distribuição temporal — quando e onde se grava mais

In [ ]:
df_time = df[df["dt_start"].notna()].copy()

if len(df_time) == 0:
    print("⚠ Timestamps não encontrados nos nomes dos arquivos.")
    print("  Verifique se o padrão FILENAME_PATTERN está correto.")
else:
    df_time["hora"]      = df_time["dt_start"].apply(lambda x: x.hour)
    df_time["dia_sem"]   = df_time["dt_start"].apply(lambda x: x.strftime("%a"))
    df_time["data"]      = df_time["dt_start"].apply(lambda x: x.date())

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Heatmap local × hora
    if df_time["local"].notna().any():
        pivot = df_time.pivot_table(
            index="local", columns="hora", values="filename",
            aggfunc="count", fill_value=0
        )
        # Garantir todas as horas
        for h in range(24):
            if h not in pivot.columns: pivot[h] = 0
        pivot = pivot[sorted(pivot.columns)]

        sns.heatmap(pivot, annot=True, fmt="d", cmap="YlOrRd",
                    linewidths=0.4, ax=axes[0],
                    cbar_kws={"label": "Nº arquivos"})
        axes[0].set_title("Gravações por Local × Hora do Dia", fontsize=12)
        axes[0].set_xlabel("Hora")
        axes[0].set_ylabel("Local")

    # Câmeras × hora
    if df_time["camera"].notna().any():
        pivot2 = df_time.pivot_table(
            index="camera", columns="hora", values="filename",
            aggfunc="count", fill_value=0
        )
        for h in range(24):
            if h not in pivot2.columns: pivot2[h] = 0
        pivot2 = pivot2[sorted(pivot2.columns)]
        sns.heatmap(pivot2, annot=True, fmt="d", cmap="Blues",
                    linewidths=0.4, ax=axes[1],
                    cbar_kws={"label": "Nº arquivos"})
        axes[1].set_title("Gravações por Câmera × Hora do Dia", fontsize=12)
        axes[1].set_xlabel("Hora")
        axes[1].set_ylabel("Câmera")

    plt.suptitle("Padrão Temporal das Gravações", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/"heatmap_local_hora.png", bbox_inches="tight")
    plt.show()

    # Tabela resumo por local
    if df["local"].notna().any():
        summary = df.groupby("local").agg(
            arquivos=("filename","count"),
            videos=("is_image", lambda x: (~x).sum()),
            imagens=("is_image", "sum"),
            tamanho_gb=("file_size_mb", lambda x: round(x.sum()/1024,2)),
            horas_gravadas=("duration_sec", lambda x: round(x.dropna().sum()/3600,1)),
            cameras=("camera","nunique"),
            resolucoes=("width", lambda x: df.loc[x.index,"width"].dropna()\
                        .astype(str).value_counts().index[0]
                        if x.notna().any() else "-"),
        ).reset_index().sort_values("horas_gravadas", ascending=False)
        print("\nResumo por local:")
        display(summary)
        summary.to_csv(OUTPUT_DIR/"summary_by_local.csv", index=False)

## 7. Análise de movimento

In [ ]:
def motion_profile(path: Path, sample_fps=1.0):
    # Abre o vídeo silenciando os logs HEVC/H264 do decoder do OpenCV
    with _suppress_stderr():
        cap = cv2.VideoCapture(str(path))
    if not cap.isOpened(): return {}
    fps   = cap.get(cv2.CAP_PROP_FPS) or 25
    skip  = max(1, int(fps / sample_fps))
    prev  = None
    scores, times = [], []
    fi = 0
    while True:
        with _suppress_stderr():
            ret, frame = cap.read()
        if not ret: break
        if fi % skip == 0:
            g = cv2.cvtColor(cv2.resize(frame,(160,90)), cv2.COLOR_BGR2GRAY)
            if prev is not None:
                scores.append(float(cv2.absdiff(g, prev).mean()))
                times.append(fi / fps)
            prev = g
        fi += 1
    cap.release()
    if not scores: return {}
    arr = np.array(scores)
    return dict(scores=scores, times=times,
                mean=float(arr.mean()), max=float(arr.max()), std=float(arr.std()),
                active_ratio=float((arr > MOTION_THRESHOLD).mean()),
                peak_sec=times[int(arr.argmax())])


print("Calculando perfil de movimento...")
motion_cache = {}
for _, row in tqdm(df_vid.iterrows(), total=len(df_vid), desc="Motion"):
    data = motion_profile(Path(row["filepath"]), MOTION_SAMPLE_FPS)
    if data:
        motion_cache[row["filename"]] = data

df["motion_mean"]  = df["filename"].map(lambda f: motion_cache.get(f,{}).get("mean"))
df["motion_max"]   = df["filename"].map(lambda f: motion_cache.get(f,{}).get("max"))
df["active_ratio"] = df["filename"].map(lambda f: motion_cache.get(f,{}).get("active_ratio"))
df["peak_sec"]     = df["filename"].map(lambda f: motion_cache.get(f,{}).get("peak_sec"))

print(f"✓ Analisados: {len(motion_cache)} vídeos")

In [ ]:
# ── Gráficos de movimento ──────────────────────────────────────────────────────
df_mot = df[df["motion_mean"].notna()].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df_mot["motion_mean"], bins=25, color="steelblue",
             edgecolor="black", alpha=0.85)
axes[0].axvline(MOTION_THRESHOLD, color="orange", lw=2, ls="--",
                label=f"Threshold ({MOTION_THRESHOLD})")
axes[0].set_title("Distribuição do Score de Movimento", fontsize=11)
axes[0].set_xlabel("Motion score médio")
axes[0].legend(fontsize=9)

axes[1].hist(df_mot["active_ratio"] * 100, bins=20, color="mediumseagreen",
             edgecolor="black", alpha=0.85)
axes[1].set_title("% do tempo com movimento", fontsize=11)
axes[1].set_xlabel("%")

if "local" in df_mot.columns and df_mot["local"].notna().any():
    df_mot.groupby("local")["motion_mean"].mean().sort_values().plot.barh(
        ax=axes[2], color="darkorange", edgecolor="black", alpha=0.85
    )
    axes[2].set_title("Score de Movimento Médio por Local", fontsize=11)

plt.suptitle("Análise de Movimento", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"motion_analysis.png", bbox_inches="tight")
plt.show()

# Top 10 mais agitados
print("\nTop 10 vídeos com MAIS movimento:")
display(df_mot.sort_values("motion_mean",ascending=False)
        [["filename","local","camera","duration_str","motion_mean","active_ratio"]].head(10))

print("\nTop 10 vídeos com MENOS movimento (possível câmera parada ou vazia):")
display(df_mot.sort_values("motion_mean",ascending=True)
        [["filename","local","camera","duration_str","motion_mean","active_ratio"]].head(10))

In [ ]:
# ── Perfil temporal de movimento de vídeos selecionados ───────────────────────
# Altere SHOW_MOTION_FOR para inspecionar vídeos específicos
SHOW_MOTION_FOR = df_mot.sort_values("motion_mean",ascending=False)["filename"].head(3).tolist()

for fname in SHOW_MOTION_FOR:
    data = motion_cache.get(fname)
    if not data: continue
    row  = df[df["filename"]==fname].iloc[0]
    t    = np.array(data["times"]) / 60
    s    = np.array(data["scores"])

    fig, ax = plt.subplots(figsize=(13, 3))
    ax.fill_between(t, s, alpha=0.3, color="steelblue")
    ax.plot(t, s, color="steelblue", lw=1.2)
    ax.axhline(MOTION_THRESHOLD, color="orange", ls="--", lw=1.5,
               label=f"Threshold ({MOTION_THRESHOLD})")
    ax.set_xlabel("Tempo (min)")
    ax.set_ylabel("Motion score")
    ax.set_title(
        f"{fname}  |  Local: {row.get('local','?')}  Câmera: {row.get('camera','?')}  "
        f"Duração: {row.get('duration_str','?')}  Ativo: {data['active_ratio']*100:.0f}% do tempo",
        fontsize=9
    )
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 8. Thumbnails visuais

In [ ]:
def show_thumbnails(path: Path, row: dict, n=MAX_THUMBS_PER_VIDEO, size=THUMB_SIZE):
    """Exibe grid de thumbnails com cabeçalho de metadados."""
    # Imagem estática
    if row.get("is_image"):
        img = open_as_image(path)
        if img is None: return
        rgb = cv2.cvtColor(cv2.resize(img, (size[0]*2, size[1]*2)), cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.imshow(rgb); ax.axis("off")
        ax.set_title(f"{path.name}  [{row['width']}×{row['height']}] — IMAGEM ESTÁTICA",
                     fontsize=9)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR/"thumbnails"/f"{path.stem}.jpg",
                    bbox_inches="tight", dpi=70)
        plt.show(); plt.close(); return

    # Vídeo
    with _suppress_stderr():
        cap = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    thumbs = []
    for idx in [int(i * total / n) for i in range(n)]:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        with _suppress_stderr():
            ret, fr = cap.read()
        if ret:
            thumbs.append(cv2.cvtColor(cv2.resize(fr, size), cv2.COLOR_BGR2RGB))
    cap.release()
    if not thumbs: return

    nc  = min(4, len(thumbs))
    nr  = math.ceil(len(thumbs) / nc)
    fig = plt.figure(figsize=(nc*4, nr*2.4+1.2))
    gs  = gridspec.GridSpec(nr+1, nc, figure=fig,
                            height_ratios=[0.55]+[1]*nr, hspace=0.3)

    ax_h = fig.add_subplot(gs[0,:])
    ax_h.axis("off")
    dur   = row.get("duration_str","?") or "?"
    mb    = row.get("file_size_mb","?")
    res   = f"{row.get('width','?')}×{row.get('height','?')}"
    mov   = f"{row.get('motion_mean',0.0):.1f}" if pd.notna(row.get("motion_mean")) else "?"
    act   = f"{row.get('active_ratio',0.0)*100:.0f}%" if pd.notna(row.get("active_ratio")) else "?"
    hdr   = (f"{path.name}\n"
             f"Local: {row.get('local','?')}  Câmera: {row.get('camera','?')}  "
             f"Início: {(row.get('ts_start','?') or '?')[:16]}  Fim: {(row.get('ts_end','?') or '?')[:16]}\n"
             f"Duração: {dur}  Tamanho: {mb} MB  Resolução: {res}  "
             f"FPS: {row.get('fps','?')}  Codec: {row.get('codec','?')}  "
             f"Movimento médio: {mov}  Ativo: {act}")
    ax_h.text(0.5, 0.5, hdr, ha="center", va="center", fontsize=8.5,
              family="monospace",
              bbox=dict(boxstyle="round", facecolor="#f0f4ff", alpha=0.9))

    for i, thumb in enumerate(thumbs):
        ax = fig.add_subplot(gs[i//nc+1, i%nc])
        ax.imshow(thumb); ax.axis("off")
        dur_s = row.get("duration_sec") or 0
        t_s   = int(dur_s * i / max(len(thumbs)-1, 1))
        ax.set_title(str(timedelta(seconds=t_s)), fontsize=8)

    plt.savefig(OUTPUT_DIR/"thumbnails"/f"{path.stem}.jpg",
                bbox_inches="tight", dpi=80)
    plt.show(); plt.close()


# ── Mostrar thumbnails — filtre conforme necessário ───────────────────────────
FILTER_LOCAL  = None    # ex: "CentroSP"  — None = todos
FILTER_CAMERA = None    # ex: "ch01"      — None = todos
MAX_SHOW      = 20      # None = todos

df_show = df.copy()
if FILTER_LOCAL:  df_show = df_show[df_show["local"]  == FILTER_LOCAL]
if FILTER_CAMERA: df_show = df_show[df_show["camera"] == FILTER_CAMERA]
if MAX_SHOW:      df_show = df_show.head(MAX_SHOW)

print(f"Exibindo {len(df_show)} arquivo(s)...")
for _, row in df_show.iterrows():
    vp = Path(row["filepath"])
    if vp.exists() and row.get("readable"):
        show_thumbnails(vp, row.to_dict())

## 9. Salvar catálogo final e perfis de movimento

In [ ]:
final_cols = ["filename","extension","local","camera","tag",
              "ts_start","ts_end","duration_str","duration_sec",
              "file_size_mb","width","height","fps","codec",
              "motion_mean","motion_max","active_ratio","peak_sec",
              "readable","is_image","pattern_matched","filepath"]
df[[c for c in final_cols if c in df.columns]].to_excel(
    OUTPUT_DIR/"catalog_full.xlsx", index=False)

# Salvar perfis de movimento (usado no nb02)
with open(OUTPUT_DIR/"motion_profiles.json", "w") as f:
    json.dump(motion_cache, f)

# Salvar catálogo como JSON (usado nos notebooks seguintes)
df.to_json(OUTPUT_DIR/"catalog.json", orient="records", force_ascii=False, indent=2)

print("✓ Arquivos salvos:")
for p in OUTPUT_DIR.iterdir():
    print(f"  {p.name:40s}  {p.stat().st_size/1024:.1f} KB")

print("\n" + "="*55)
print(" PRÓXIMO PASSO: 02_client_classification.ipynb")
print("="*55)